In [ ]:
%matplotlib widget

import numpy as np
import mpmath as mp
import matplotlib.pyplot as plt
from ipywidgets import IntSlider, FloatSlider, VBox, HBox, Layout, HTML
from IPython.display import display

plt.ioff()

# ==============================================================================
# USAGE
#
# This interactive notebook visualizes ALL 2N poles of the function
#
#       H(s)H(-s)
#
# associated with a continuous-time elliptic / Cauer low-pass filter.
#
# Three parameters can be varied:
#
#       N     : filter order
#       ωs    : stopband-edge angular frequency
#       Ap    : maximum passband attenuation in dB
#
# The prototype passband edge is fixed at:
#
#       ωp = 1 rad/s
#
# so that:
#
#       k2 = ωp / ωs
#
# The pole equations are evaluated directly from the theoretical relations
# developed for elliptic filters.
#
# The calculation proceeds through:
#
#       K2
#       p0
#       Ωm
#       Vm
#       W
#
# For odd N:
#
#       one stable pole is real and negative:
#
#           p0
#
#       and the remaining N-1 poles occur in complex-conjugate pairs.
#
# For even N:
#
#       there is no real pole and all N stable poles occur in
#       complex-conjugate pairs.
#
# Pole representation:
#
#       Filled red circles : poles used in the stable transfer function H(s)
#       Open red circles   : corresponding poles of H(-s), rejected because
#                            Re{p} > 0
#
# Only the N poles in the left half-plane are retained in H(s).
#
# IMPORTANT:
#
# The stopband attenuation As is not used as an independent slider here.
# Once N is explicitly selected, the pole equations depend on Ap and on the
# frequency ratio k2 = ωp/ωs. As enters the minimum-order design problem, not
# the pole-position equations once N is prescribed.
#
# This notebook intentionally displays only poles, not transmission zeros.
# ==============================================================================

# ==============================================================================
# DISPLAY SETTINGS
# ==============================================================================

display(HTML("""
<style>
.jp-OutputArea,
.jp-OutputArea-child,
.jp-OutputArea-output {
    overflow: visible !important;
    max-height: none !important;
    height: auto !important;
}

.output,
.output_area,
.output_subarea,
.output_scroll {
    overflow: visible !important;
    max-height: none !important;
    height: auto !important;
}

.jupyter-widgets,
.widget-box,
.widget-html,
.widget-html-content {
    overflow: visible !important;
    max-height: none !important;
}

.jp-Cell-outputWrapper {
    overflow: visible !important;
}
</style>
"""))

# ==============================================================================
# FIXED NORMALIZED PASSBAND EDGE
# ==============================================================================

wp = 1.0

# ==============================================================================
# DESCRIPTION
# ==============================================================================

description = HTML("""
<div style="
    border:1px solid #9ec9f5;
    border-radius:7px;
    padding:8px 10px;
    margin:0px 0px 8px 0px;
    font-size:13px;
    line-height:1.45;
    background-color:#f7fbff;
    width:1040px;
    max-width:1040px;
    box-sizing:border-box;
">
<b>Purpose:</b>
Visualize all 2N poles of H(s)H(-s) for an elliptic/Cauer low-pass filter
and identify the N poles that form the stable transfer function H(s).
<br>
<b>Interpretation:</b>
The pole locations are calculated from the elliptic-filter pole equations
using the auxiliary quantities p<sub>0</sub>, Ω<sub>m</sub>,
V<sub>m</sub> and W. The normalized passband edge is fixed at
ω<sub>p</sub> = 1 rad/s. Filled red circles represent the poles in the
left half-plane that are retained in H(s), while open red circles represent
their right-half-plane counterparts belonging to H(-s).
</div>
""", layout=Layout(width='1050px', max_width='1050px'))

# ==============================================================================
# CONTROLS
# ==============================================================================

slider_layout = Layout(width='280px')

style_opts = {'description_width':'95px'}

order_slider = IntSlider(min=2, max=10, step=1, value=5, description='Order N:', continuous_update=True, style=style_opts, layout=slider_layout)

ws_slider = FloatSlider(min=1.1, max=5.0, step=0.1, value=2.0, description='ωs:', continuous_update=True, readout=True, readout_format='.1f', style=style_opts, layout=slider_layout)

Ap_slider = FloatSlider(min=0.1, max=3.0, step=0.1, value=1.0, description='Ap (dB):', continuous_update=True, readout=True, readout_format='.1f', style=style_opts, layout=slider_layout)

parameter_title = HTML("""
<div style="
    font-size:14px;
    font-weight:bold;
    margin-top:3px;
    margin-bottom:5px;
">
Filter Parameters:
</div>
""")

info_html = HTML(layout=Layout(width='370px', max_width='370px'))

pole_table = HTML(layout=Layout(width='460px', max_width='460px'))

# ==============================================================================
# FIGURE
# ==============================================================================

fig, ax = plt.subplots(figsize=(6.6, 6.6))

used_scatter = ax.scatter([], [], s=95, marker='o', facecolors='red', edgecolors='red', linewidths=1.5, label='Used poles')

rejected_scatter = ax.scatter([], [], s=95, marker='o', facecolors='white', edgecolors='red', linewidths=1.8, label='Rejected poles')

ax.axhline(0.0, color='black', linewidth=0.9)

ax.axvline(0.0, color='black', linewidth=0.9)

ax.set_xlabel('Re{s}', fontsize=11)

ax.set_ylabel('Im{s}', fontsize=11)

ax.set_title('Elliptic Filter Poles of H(s)H(-s)', fontsize=13, fontweight='bold', pad=8)

ax.grid(True, linestyle=':', alpha=0.35)

ax.set_aspect('equal', adjustable='box')

ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.10), ncol=2, fontsize=8)

# ==============================================================================
# FIXED AXIS LIMITS
#
# The pole locations for the selected parameter ranges remain close to the
# origin. Therefore the fixed interval [-2, 2] is used to make the geometry
# much easier to observe.
# ==============================================================================

axis_limit = 2.0

ax.set_xlim(-axis_limit, axis_limit)

ax.set_ylim(-axis_limit, axis_limit)

ax.set_xticks(np.arange(-2.0, 2.1, 0.5))

ax.set_yticks(np.arange(-2.0, 2.1, 0.5))

fig.subplots_adjust(left=0.12, right=0.96, bottom=0.17, top=0.92)

fig.canvas.header_visible = False

fig.canvas.toolbar_visible = False

fig.canvas.resizable = False

fig.canvas.layout.width = '660px'

fig.canvas.layout.height = '660px'

# ==============================================================================
# JACOBI ELLIPTIC SINE
#
# mpmath.ellipfun uses the elliptic parameter:
#
#       m = k²
#
# whereas the theoretical equations use the elliptic modulus k.
# ==============================================================================

def jacobi_sn(u, k):

    return complex(mp.ellipfun('sn', u, k**2))

# ==============================================================================
# COMPLETE ELLIPTIC INTEGRAL K(k)
#
# mpmath.ellipk also uses the parameter:
#
#       m = k²
# ==============================================================================

def complete_K(k):

    return float(mp.ellipk(k**2))

# ==============================================================================
# ELLIPTIC POLE CALCULATION
# ==============================================================================

def elliptic_stable_poles(N, wp, ws, Ap):

    # --------------------------------------------------------------------------
    # Elliptic measure
    #
    #       k2 = ωp / ωs
    # --------------------------------------------------------------------------

    k2 = wp / ws

    # --------------------------------------------------------------------------
    # Complete elliptic integral
    #
    #       K2 = K(k2)
    # --------------------------------------------------------------------------

    K2 = complete_K(k2)

    # --------------------------------------------------------------------------
    # Auxiliary logarithmic quantity
    #
    #       L = ln[(10^(Ap/20)+1)/(10^(Ap/20)-1)]
    # --------------------------------------------------------------------------

    A = 10.0**(Ap / 20.0)

    logarithmic_term = np.log((A + 1.0) / (A - 1.0))

    # --------------------------------------------------------------------------
    # Imaginary Jacobi argument used for p0
    #
    #               K2
    #       u0 = j ------ L
    #              Nπ
    # --------------------------------------------------------------------------

    u0 = 1j * K2 * logarithmic_term / (N * np.pi)

    # --------------------------------------------------------------------------
    # Auxiliary real parameter p0
    #
    #       p0 = j ωp sn(u0,k2)
    #
    # Although p0 is an actual pole only for odd N, the same parameter is
    # required in the pole equations for even N.
    # --------------------------------------------------------------------------

    p0_complex = 1j * wp * jacobi_sn(u0, k2)

    p0 = float(np.real_if_close(p0_complex, tol=1000).real)

    # --------------------------------------------------------------------------
    # Auxiliary parameter W
    #
    #       W = sqrt[(1+p0²/ωp²)(1+p0²/ωs²)]
    # --------------------------------------------------------------------------

    W = np.sqrt((1.0 + p0**2 / wp**2) * (1.0 + p0**2 / ws**2))

    # --------------------------------------------------------------------------
    # Stable pole list
    # --------------------------------------------------------------------------

    stable_poles = []

    Omega_values = []

    V_values = []

    # ==========================================================================
    # ODD ORDER
    # ==========================================================================

    if N % 2 == 1:

        # ----------------------------------------------------------------------
        # Real negative pole
        # ----------------------------------------------------------------------

        stable_poles.append(complex(p0, 0.0))

        # ----------------------------------------------------------------------
        # Complex-conjugate pole pairs
        #
        #                  1       [2m K2]
        #       Ωm = ----------- sn[------,k2]
        #                ωs          N
        #
        #       m = 1,...,(N-1)/2
        # ----------------------------------------------------------------------

        m_values = range(1, (N - 1) // 2 + 1)

        for m in m_values:

            argument = 2.0 * m * K2 / N

            Omega_m = jacobi_sn(argument, k2).real / ws

            V_m = np.sqrt(max(0.0, (1.0 - wp**2 * Omega_m**2) * (1.0 - ws**2 * Omega_m**2)))

            denominator = 1.0 + p0**2 * Omega_m**2

            sigma_m = p0 * V_m / denominator

            omega_m = wp * ws * Omega_m * W / denominator

            stable_poles.append(complex(sigma_m, +omega_m))

            stable_poles.append(complex(sigma_m, -omega_m))

            Omega_values.append(Omega_m)

            V_values.append(V_m)

    # ==========================================================================
    # EVEN ORDER
    # ==========================================================================

    else:

        # ----------------------------------------------------------------------
        # No real pole.
        #
        #                  1       [(2m-1)K2]
        #       Ωm = ----------- sn[----------,k2]
        #                ωs             N
        #
        #       m = 1,...,N/2
        # ----------------------------------------------------------------------

        m_values = range(1, N // 2 + 1)

        for m in m_values:

            argument = (2.0 * m - 1.0) * K2 / N

            Omega_m = jacobi_sn(argument, k2).real / ws

            V_m = np.sqrt(max(0.0, (1.0 - wp**2 * Omega_m**2) * (1.0 - ws**2 * Omega_m**2)))

            denominator = 1.0 + p0**2 * Omega_m**2

            sigma_m = p0 * V_m / denominator

            omega_m = wp * ws * Omega_m * W / denominator

            stable_poles.append(complex(sigma_m, +omega_m))

            stable_poles.append(complex(sigma_m, -omega_m))

            Omega_values.append(Omega_m)

            V_values.append(V_m)

    return np.asarray(stable_poles, dtype=complex), k2, K2, logarithmic_term, p0, W, np.asarray(Omega_values), np.asarray(V_values)

# ==============================================================================
# UPDATE FUNCTION
# ==============================================================================

def update_elliptic_poles(change=None):

    N = order_slider.value

    ws = ws_slider.value

    Ap = Ap_slider.value

    # --------------------------------------------------------------------------
    # Calculate the N stable elliptic-filter poles
    # --------------------------------------------------------------------------

    used_poles, k2, K2, logarithmic_term, p0, W, Omega_values, V_values = elliptic_stable_poles(N, wp, ws, Ap)

    # --------------------------------------------------------------------------
    # H(s)H(-s)
    #
    # If pk is a pole of H(s), then -pk is the corresponding pole of H(-s).
    # --------------------------------------------------------------------------

    rejected_poles = -used_poles

    poles = np.concatenate((used_poles, rejected_poles))

    # --------------------------------------------------------------------------
    # Update pole locations
    # --------------------------------------------------------------------------

    used_offsets = np.column_stack((np.real(used_poles), np.imag(used_poles)))

    rejected_offsets = np.column_stack((np.real(rejected_poles), np.imag(rejected_poles)))

    used_scatter.set_offsets(used_offsets)

    rejected_scatter.set_offsets(rejected_offsets)

    # --------------------------------------------------------------------------
    # Pole table
    # --------------------------------------------------------------------------

    rows = ""

    for index, pole in enumerate(poles):

        if index < N:

            status = "USED"

            status_style = """
                color:#0066cc;
                background:#eef6ff;
                border:1px solid #9bc8f5;
            """

        else:

            status = "REJECTED"

            status_style = """
                color:#cc0000;
                background:#fff1f1;
                border:1px solid #efaaaa;
            """

        rows += f"""
        <tr style="border-bottom:1px solid #eeeeee;">

            <td style="
                padding:5px 8px;
                text-align:center;
                font-family:'Times New Roman',serif;
                font-size:17px;
                font-style:italic;
                white-space:nowrap;
            ">
                p<sub>{index}</sub>
            </td>

            <td style="
                padding:5px 10px;
                font-family:'Times New Roman',serif;
                font-size:16px;
                white-space:nowrap;
            ">
                {pole.real:+.6f}
                <span style="font-style:italic;">{pole.imag:+.6f}j</span>
            </td>

            <td style="
                padding:5px 8px;
                text-align:center;
            ">
                <span style="
                    {status_style}
                    padding:2px 7px;
                    border-radius:10px;
                    font-size:10px;
                    font-weight:bold;
                    letter-spacing:0.3px;
                    white-space:nowrap;
                ">
                    {status}
                </span>
            </td>

        </tr>
        """

    pole_table.value = f"""
    <div style="
        border:1px solid #cccccc;
        border-radius:7px;
        padding:8px;
        background:white;
        width:450px;
        max-height:620px;
        overflow-y:auto;
        box-sizing:border-box;
        font-size:12px;
    ">

    <div style="
        font-family:'Times New Roman',serif;
        font-size:18px;
        font-weight:bold;
        margin-bottom:7px;
        text-align:center;
    ">
        Pole Values
    </div>

    <table style="
        width:100%;
        border-collapse:collapse;
    ">

        <tr style="border-bottom:1px solid #bbbbbb;">
            <th style="padding:5px;">Pole</th>
            <th style="padding:5px;">Complex Value</th>
            <th style="padding:5px;">Status</th>
        </tr>

        {rows}

    </table>

    </div>
    """

    # --------------------------------------------------------------------------
    # Information panel
    # --------------------------------------------------------------------------

    if N % 2 == 1:

        parity_text = "Odd order"

        real_candidates = used_poles[np.abs(np.imag(used_poles)) < 1e-8]

        if len(real_candidates) > 0:

            real_pole_text = f"One stable real pole at s = {real_candidates[0].real:.6f}"

        else:

            real_pole_text = "One stable real pole"

    else:

        parity_text = "Even order"

        real_pole_text = "No real pole"

    # --------------------------------------------------------------------------
    # Auxiliary values for display
    # --------------------------------------------------------------------------

    if len(Omega_values) > 0:

        Omega_text = ", ".join([f"{value:.5f}" for value in Omega_values])

        V_text = ", ".join([f"{value:.5f}" for value in V_values])

    else:

        Omega_text = "—"

        V_text = "—"

    info_html.value = f"""
    <div style="
        border:1px solid #cccccc;
        border-radius:7px;
        padding:8px 10px;
        margin-top:8px;
        font-size:12px;
        line-height:1.65;
        background:white;
        width:365px;
        box-sizing:border-box;
    ">

    <div>
        <b>Filter:</b>
        <span style="color:#0066cc;">Elliptic / Cauer low-pass</span>
    </div>

    <div>
        <b>Order N:</b>
        <span style="color:#0066cc;">{N}</span>
    </div>

    <div>
        <b>Passband edge:</b>
        <span style="color:#0066cc;">ωp = {wp:.2f} rad/s</span>
    </div>

    <div>
        <b>Stopband edge:</b>
        <span style="color:#0066cc;">ωs = {ws:.2f} rad/s</span>
    </div>

    <div>
        <b>Passband attenuation:</b>
        <span style="color:#0066cc;">Ap = {Ap:.1f} dB</span>
    </div>

    <div>
        <b>Elliptic measure:</b>
        <span style="color:#0066cc;">k₂ = ωp/ωs = {k2:.6f}</span>
    </div>

    <div>
        <b>Complete elliptic integral:</b>
        <span style="color:#0066cc;">K₂ = {K2:.6f}</span>
    </div>

    <div style="
        margin-top:5px;
        padding-top:5px;
        border-top:1px solid #eeeeee;
    ">
        <b>Auxiliary pole parameters:</b>
    </div>

    <div>
        <b>p₀:</b>
        <span style="color:#0066cc;">{p0:.6f}</span>
    </div>

    <div>
        <b>W:</b>
        <span style="color:#0066cc;">{W:.6f}</span>
    </div>

    <div>
        <b>Ωm:</b>
        <span style="color:#0066cc;">{Omega_text}</span>
    </div>

    <div>
        <b>Vm:</b>
        <span style="color:#0066cc;">{V_text}</span>
    </div>

    <div style="
        margin-top:5px;
        padding-top:5px;
        border-top:1px solid #eeeeee;
    ">
        <b>Total poles:</b>
        <span style="color:#0066cc;">{2 * N}</span>
    </div>

    <div>
        <b>Used poles:</b>
        <span style="color:#0066cc;">{N}</span>
    </div>

    <div>
        <b>Rejected poles:</b>
        <span style="color:#0066cc;">{N}</span>
    </div>

    <div>
        <b>Order type:</b>
        <span style="color:#0066cc;">{parity_text}</span>
    </div>

    <div>
        <b>Real pole:</b>
        <span style="color:#0066cc;">{real_pole_text}</span>
    </div>

    <div style="
        margin-top:6px;
        padding-top:6px;
        border-top:1px solid #eeeeee;
    ">
        <b>Observation:</b><br>
        The elliptic-filter poles are determined through Jacobi elliptic
        functions. For odd order, one negative real pole p₀ appears together
        with complex-conjugate pairs. For even order, all stable poles occur
        in complex-conjugate pairs. Changing ωs changes k₂ and therefore the
        elliptic geometry itself; changing Ap modifies p₀ and consequently
        the positions of all pole pairs.
    </div>

    </div>
    """

    # --------------------------------------------------------------------------
    # Redraw
    # --------------------------------------------------------------------------

    fig.canvas.draw_idle()

# ==============================================================================
# CALLBACKS
# ==============================================================================

order_slider.observe(update_elliptic_poles, names='value')

ws_slider.observe(update_elliptic_poles, names='value')

Ap_slider.observe(update_elliptic_poles, names='value')

# ==============================================================================
# LAYOUT
# ==============================================================================

controls = VBox([parameter_title, order_slider, ws_slider, Ap_slider, info_html], layout=Layout(width='380px', min_width='380px', max_width='380px', flex='0 0 380px', align_items='flex-start'))

main_row = HBox([controls, fig.canvas, pole_table], layout=Layout(width='1510px', align_items='flex-start', justify_content='flex-start'))

# ==============================================================================
# INITIALIZE
# ==============================================================================

update_elliptic_poles()

# ==============================================================================
# DISPLAY
# ==============================================================================

display(description)

display(main_row)